# Sandbox: Check DICOM RescaleSlope / RescaleIntercept for velocity series

In [6]:
import platform
from pathlib import Path

import pydicom
import pandas as pd

from vascular_superenhancement.utils.path_config import load_path_config, _PROJECT_ROOT

config_name = "local_mac" if platform.system() == "Darwin" else "all_patients"
print(config_name)
pc = load_path_config(config_name)

WORKING_DIR = pc.working_dir
PATIENT_DATA_DIR = WORKING_DIR / "patient_data"
REPO_ROOT = _PROJECT_ROOT

splits_df = pd.read_csv(REPO_ROOT / "splits" / "splits_01-15-26.csv")
test_patients = sorted(splits_df.loc[splits_df["split"] == "test", "patient_id"].tolist())

# Pick 3 patients to check
check_patients = test_patients[:3]
print(f"Checking: {check_patients}")

local_mac
Checking: ['Balboloop', 'Biswifo', 'Bomatog']


In [7]:
# Component codes: 3=vx, 4=vy, 5=vz (velocity), 2=magnitude
COMP_NAMES = {2: "magnitude", 3: "vx", 4: "vy", 5: "vz"}

# Catalog stores Linux absolute paths — remap to Mac mount
LINUX_PREFIX = "/home/ayeluru/mnt/fourier"
MAC_PREFIX = str(pc.base_data_dir)  # /Volumes

def remap_path(p: str) -> Path:
    return Path(p.replace(LINUX_PREFIX, MAC_PREFIX))

for pid in check_patients:
    print(f"\n{'='*60}")
    print(f"Patient: {pid}")
    print(f"{'='*60}")

    # Load catalog directly from CSV (avoids Patient logger creating dirs on NAS)
    catalog_path = PATIENT_DATA_DIR / pid / f"dicom_catalog_4d-flow_{pid}.csv"
    if not catalog_path.exists():
        print(f"  Catalog not found: {catalog_path}")
        continue
    catalog = pd.read_csv(catalog_path)

    # Check one DICOM per component (first timepoint, first slice)
    for code, name in COMP_NAMES.items():
        comp_rows = catalog[catalog["tag_0x0043_0x1030"] == code]
        if comp_rows.empty:
            print(f"  {name} (code={code}): no entries in catalog")
            continue

        dcm_path = remap_path(comp_rows.iloc[0]["filepath"])
        if not dcm_path.exists():
            print(f"  {name} (code={code}): file not found: {dcm_path}")
            continue

        ds = pydicom.dcmread(dcm_path, stop_before_pixels=True)

        slope = getattr(ds, "RescaleSlope", None)
        intercept = getattr(ds, "RescaleIntercept", None)
        bits_stored = getattr(ds, "BitsStored", None)
        bits_alloc = getattr(ds, "BitsAllocated", None)
        pixel_repr = getattr(ds, "PixelRepresentation", None)

        print(f"  {name} (code={code}):")
        print(f"    RescaleSlope     = {slope}")
        print(f"    RescaleIntercept = {intercept}")
        print(f"    BitsStored       = {bits_stored}")
        print(f"    BitsAllocated    = {bits_alloc}")
        print(f"    PixelRepresentation = {pixel_repr} ({'signed' if pixel_repr == 1 else 'unsigned' if pixel_repr == 0 else '?'})")
        print(f"    File: {dcm_path.name}")


Patient: Balboloop
  magnitude (code=2):
    RescaleSlope     = None
    RescaleIntercept = None
    BitsStored       = 16
    BitsAllocated    = 16
    PixelRepresentation = 1 (signed)
    File: 2.25.100354442864893407636410556130220117080.dcm
  vx (code=3):
    RescaleSlope     = None
    RescaleIntercept = None
    BitsStored       = 16
    BitsAllocated    = 16
    PixelRepresentation = 1 (signed)
    File: 2.25.100021239444098341308854231024982682225.dcm
  vy (code=4):
    RescaleSlope     = None
    RescaleIntercept = None
    BitsStored       = 16
    BitsAllocated    = 16
    PixelRepresentation = 1 (signed)
    File: 2.25.100000403025816259866829127508564906572.dcm
  vz (code=5):
    RescaleSlope     = None
    RescaleIntercept = None
    BitsStored       = 16
    BitsAllocated    = 16
    PixelRepresentation = 1 (signed)
    File: 2.25.100004543710420146343249015962758457137.dcm

Patient: Biswifo
  magnitude (code=2):
    RescaleSlope     = None
    RescaleIntercept = None
 